In [3]:
eval_dataset = [
    {
        "inputs":{"question":"What is the capital of France?"},
        "expectations":{"expected_response":"Paris"},
    },
    {
        "inputs":{"question":"Who was the first person to land on the moon?"},
        "expectations":{"expected_response":"Neil Armstrong"},
    }
]


In [ ]:
from mlflow.genai import scorer
from mlflow.genai.scorers import Correctness, Guidelines
from src.llm_model import LLMChat
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

@scorer
def is_consice(outputs: str) -> bool:
    return len(outputs) <= 10

scorers = [
    Correctness(),
    Guidelines(name="is_english", guidelines="The answer must be in English"),
    is_consice,
]
chat = LLMChat()
def qa_predict_fn(question:str) -> str:
    return chat.llm_chat_no_tools(question)

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=scorers,
    predict_fn=qa_predict_fn,
)


2026/10/26 12:17:28 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:03, Remaining: 00:00] [predict_fn: 91%, scorers: 9%]
2026/10/26 12:17:33 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'correctness': 2/2 failed, 'is_english': 2/2 failed. Check individual trace assessments for detailed error messages.


In [10]:
import mlflow
from langchain_ollama import ChatOllama
from src.llm_model import LLMChat
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://localhost:5000")

# Specify the experiment you just created for your LLM application or AI agent.
# The positional argument is the experiment name. To attach to an existing
# experiment by its numeric ID, pass it with the experiment_id keyword (a numeric
# value passed positionally is treated as a name, not an ID):
mlflow.set_experiment("langchain test")
# mlflow.set_experiment(experiment_id="1234567890123456")

# Enable automatic tracing for all OpenAI API calls.
mlflow.langchain.autolog()

client = LLMChat()
await client.llm_chat('what are your available tools?')
await client.llm_chat("write a function that applies graph coloring based on dfs")



[AGENT]: I have three main tools available for working with scripts:

1. **write_script_tool** - Save a Python function tool with typed parameters, return type and docstring. This creates new scripts that need to be reviewed and enabled before execution.

2. **search_scripts** - Search saved script names and descriptions; return IDs, schemas and status. This helps you find existing scripts in the registry.

3. **execute_script** - Execute an enabled registered function by ID with JSON arguments. Returns its JSON result and captured stdout. Execution has a ten-second timeout.

These tools work together to help you:
- Create new Python scripts (write_script_tool)
- Find existing scripts in your registry (search_scripts)
- Run enabled scripts with specific arguments (execute_script)

Would you like me to search for any existing scripts, or would you like help creating a new script?
[TOOL CALL]: write_script_tool
[OBS]: {'error': 'Use JSON-compatible annotations: str, int, float, bool, lis

[SystemMessage(content="You are a voice assistant with a persistent Python tool registry.\nSearch with search_scripts before creating a tool. Use execute_script with an\nexisting enabled tool's ID and schema-compatible arguments when it fits.\nIf no suitable tool exists, use write_script_tool to save a new script. Give\nits entry-point function typed named parameters, a return annotation, a clear\ndocstring, and a JSON-serializable result. script_name has no path or extension.\nNew scripts are pending_validation: explain that review and enabling are needed\nbefore execution. Never claim a pending script has run. Tool failures are\nreturned as error objects; read them and correct the request when appropriate.\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what are your available tools?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I have three main tools available for working with scripts:\n\n1. **write_script_tool** - Save a Python function

[Trace(trace_id='tr-4273791b8a8c6f11381e1527f214a245'), Trace(trace_id='tr-f04b0ae039ef612a4da66488166a29ef'), Trace(trace_id='tr-fe22270179e679d37a76edf90b09981e')]